# Stage 2 — Hyperparameter Study

Notebook này dùng để nghiên cứu ảnh hưởng của tham số trong từng phương pháp denoising.

Mục tiêu:
- Không chỉ so method với method.
- Hiểu vì sao tham số thay đổi làm ảnh mượt hơn, mất chi tiết hơn, PSNR tăng/giảm, runtime tăng/giảm.
- Tạo dữ liệu và hình ảnh để đưa vào report/thuyết trình.

Các nhóm tham số:
- **Gaussian**: `kernel_size`, `sigma_filter`
- **NLM**: `patch_size`, `search_size`, `h`
- **BM3D**: `sigma_psd`
- **K-SVD**: `patch_size`, `dict_size`, `sparsity`, `iterations` — mặc định tắt vì chậm


In [ ]:
from pathlib import Path
import sys
import time
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io, color, img_as_ubyte

try:
    ROOT = Path.cwd()
    if ROOT.name.lower() == "notebooks":
        ROOT = ROOT.parent
except Exception:
    ROOT = Path(".").resolve()

SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)
print("SRC :", SRC)


In [ ]:
from noise import add_gaussian_noise
from filters import gaussian_filter
from metrics import psnr, ssim

try:
    from nlm import nlm_denoise_fast
    HAS_NLM = True
except Exception as e:
    print("Cannot import NLM:", e)
    HAS_NLM = False

try:
    from bm3d_wrapper import bm3d_denoise
    HAS_BM3D = True
except Exception as e:
    print("Cannot import BM3D:", e)
    HAS_BM3D = False

try:
    from ksvd_denoising import ksvd_denoise
    HAS_KSVD = True
except Exception as e:
    print("Cannot import K-SVD:", e)
    HAS_KSVD = False

print("HAS_NLM :", HAS_NLM)
print("HAS_BM3D:", HAS_BM3D)
print("HAS_KSVD:", HAS_KSVD)


## 1. Cấu hình thí nghiệm

Khuyên dùng:
- `DATASET = "set12"`
- `IMAGE_NAME = "01"` hoặc một ảnh Set12 cụ thể
- `SIGMA = 25`
- `RUN_KSVD = False` lúc đầu, sau khi mọi thứ ổn mới đổi thành `True`


In [ ]:
DATASET = "set12"
IMAGE_NAME = "01"
SIGMA = 25
SEED = 42

# K-SVD chậm, nên mặc định tắt.
RUN_KSVD = False

OUT_DIR = ROOT / "results" / "hyperparams"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Output:", OUT_DIR)


In [ ]:
def read_grayscale_uint8(path):
    img = io.imread(path)
    if img.ndim == 3:
        img = color.rgb2gray(img)
        img = img_as_ubyte(img)
    else:
        if img.dtype != np.uint8:
            img = img_as_ubyte(img)
    return img.astype(np.uint8)


def get_dataset_folder(dataset):
    dataset = dataset.lower()
    if dataset == "set12":
        return ROOT / "data" / "Set12"
    if dataset == "bsd68":
        return ROOT / "data" / "BSD68"
    raise ValueError("dataset must be set12 or bsd68")


def find_image_path(dataset, image_name):
    folder = get_dataset_folder(dataset)
    candidates = []
    for ext in [".png", ".jpg", ".jpeg", ".bmp"]:
        candidates.append(folder / f"{image_name}{ext}")
        if str(image_name).isdigit():
            candidates.append(folder / f"{int(image_name):02d}{ext}")

    for path in candidates:
        if path.exists():
            return path

    available = sorted([p.stem for p in folder.glob("*") if p.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]])
    raise FileNotFoundError(f"Image '{image_name}' not found. Available examples: {available[:20]}")


image_path = find_image_path(DATASET, IMAGE_NAME)
clean = read_grayscale_uint8(image_path)
noisy = add_gaussian_noise(clean, sigma=SIGMA, seed=SEED)

print("Image:", image_path)
print("Clean shape:", clean.shape)
print("Noisy PSNR:", psnr(clean, noisy))
print("Noisy SSIM:", ssim(clean, noisy))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(clean, cmap="gray", vmin=0, vmax=255)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(noisy, cmap="gray", vmin=0, vmax=255)
plt.title(f"Noisy σ={SIGMA}\nPSNR={psnr(clean, noisy):.2f} dB")
plt.axis("off")

plt.tight_layout()
plt.show()


## 2. Hàm chạy một method với một bộ tham số

In [ ]:
def run_method(clean, noisy, method_name, param_dict):
    t0 = time.perf_counter()

    if method_name == "Gaussian":
        denoised = gaussian_filter(
            noisy,
            kernel_size=int(param_dict["kernel_size"]),
            sigma=float(param_dict["sigma_filter"]),
        )

    elif method_name == "NLM":
        denoised = nlm_denoise_fast(
            noisy,
            patch_size=int(param_dict["patch_size"]),
            search_size=int(param_dict["search_size"]),
            h=float(param_dict["h"]),
        )

    elif method_name == "BM3D":
        denoised = bm3d_denoise(
            noisy,
            sigma_psd=float(param_dict["sigma_psd"]),
        )

    elif method_name == "K-SVD":
        denoised, _, _ = ksvd_denoise(
            noisy,
            patch_size=int(param_dict["patch_size"]),
            dict_size=int(param_dict["dict_size"]),
            sparsity=int(param_dict["sparsity"]),
            iterations=int(param_dict["iterations"]),
        )

    else:
        raise ValueError(f"Unknown method: {method_name}")

    runtime_ms = (time.perf_counter() - t0) * 1000
    denoised = np.clip(denoised, 0, 255).astype(np.uint8)

    return {
        "image": denoised,
        "psnr": psnr(clean, denoised),
        "ssim": ssim(clean, denoised),
        "runtime_ms": runtime_ms,
    }


## 3. Tạo grid tham số

Bạn có thể giảm grid nếu máy chạy chậm.

In [ ]:
experiments = []

# Gaussian: blur mạnh/yếu
for kernel_size, sigma_filter in product([3, 5, 7, 9], [0.5, 1.0, 1.5, 2.0, 3.0]):
    experiments.append({
        "method": "Gaussian",
        "params": {
            "kernel_size": kernel_size,
            "sigma_filter": sigma_filter,
        }
    })

# NLM: patch similarity và search window
if HAS_NLM:
    for patch_size, search_size, h_factor in product([3, 5, 7], [11, 21, 31], [0.6, 0.8, 1.0, 1.2, 1.5]):
        experiments.append({
            "method": "NLM",
            "params": {
                "patch_size": patch_size,
                "search_size": search_size,
                "h": SIGMA * h_factor,
                "h_factor": h_factor,
            }
        })

# BM3D: cố tình đưa sigma_psd thấp/đúng/cao để xem under-denoise/over-smooth
if HAS_BM3D:
    for sigma_factor in [0.5, 0.75, 1.0, 1.25, 1.5]:
        experiments.append({
            "method": "BM3D",
            "params": {
                "sigma_psd": SIGMA * sigma_factor,
                "sigma_factor": sigma_factor,
            }
        })

# K-SVD: tắt mặc định vì chậm
if RUN_KSVD and HAS_KSVD:
    for patch_size, dict_size, sparsity, iterations in product([6, 8], [64, 128], [2, 3, 4], [3, 5]):
        experiments.append({
            "method": "K-SVD",
            "params": {
                "patch_size": patch_size,
                "dict_size": dict_size,
                "sparsity": sparsity,
                "iterations": iterations,
            }
        })

print("Total experiments:", len(experiments))
pd.DataFrame([{"method": e["method"], **e["params"]} for e in experiments]).head(20)


## 4. Chạy hyperparameter study

In [ ]:
records = []
best_images = {}

for i, exp in enumerate(experiments, start=1):
    method = exp["method"]
    params = exp["params"]

    print(f"[{i}/{len(experiments)}] {method} | {params}", end="")

    try:
        result = run_method(clean, noisy, method, params)

        record = {
            "dataset": DATASET,
            "image": image_path.stem,
            "sigma": SIGMA,
            "seed": SEED,
            "method": method,
            "psnr": result["psnr"],
            "ssim": result["ssim"],
            "runtime_ms": result["runtime_ms"],
        }
        record.update(params)
        records.append(record)

        if method not in best_images or result["psnr"] > best_images[method]["psnr"]:
            best_images[method] = {
                "image": result["image"],
                "psnr": result["psnr"],
                "ssim": result["ssim"],
                "runtime_ms": result["runtime_ms"],
                "params": params,
            }

        print(f" -> PSNR={result['psnr']:.2f}, SSIM={result['ssim']:.4f}, time={result['runtime_ms']:.1f} ms")

    except Exception as e:
        print(" -> ERROR:", e)
        record = {
            "dataset": DATASET,
            "image": image_path.stem,
            "sigma": SIGMA,
            "seed": SEED,
            "method": method,
            "psnr": np.nan,
            "ssim": np.nan,
            "runtime_ms": np.nan,
            "error": str(e),
        }
        record.update(params)
        records.append(record)

df = pd.DataFrame(records)
df.head()


## 5. Lưu CSV kết quả

In [ ]:
raw_path = OUT_DIR / f"{DATASET}_{image_path.stem}_sigma{SIGMA}_hyperparams_raw.csv"
summary_path = OUT_DIR / f"{DATASET}_{image_path.stem}_sigma{SIGMA}_hyperparams_top5.csv"

df.to_csv(raw_path, index=False)

top5 = (
    df.dropna(subset=["psnr"])
      .sort_values(["method", "psnr"], ascending=[True, False])
      .groupby("method", as_index=False)
      .head(5)
)

top5.to_csv(summary_path, index=False)

print("Saved raw    :", raw_path)
print("Saved top 5  :", summary_path)

top5


## 6. Xem bộ tham số tốt nhất từng method

In [ ]:
best = (
    df.dropna(subset=["psnr"])
      .sort_values("psnr", ascending=False)
      .groupby("method", as_index=False)
      .first()
      .sort_values("psnr", ascending=False)
)

best


## 7. Biểu đồ Best PSNR theo method

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(best["method"], best["psnr"])
plt.ylabel("Best PSNR (dB)")
plt.title(f"Best Hyperparameter Result — {image_path.stem}, σ={SIGMA}")
plt.xticks(rotation=30)
plt.tight_layout()

fig_path = FIG_DIR / f"{DATASET}_{image_path.stem}_sigma{SIGMA}_best_psnr.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

fig_path


## 8. Biểu đồ PSNR vs Runtime

Biểu đồ này rất hữu ích để nói trong báo cáo: tham số tốt về chất lượng chưa chắc tốt về thời gian chạy.

In [ ]:
valid = df.dropna(subset=["psnr", "runtime_ms"])

plt.figure(figsize=(7, 5))
for method in valid["method"].unique():
    sub = valid[valid["method"] == method]
    plt.scatter(sub["runtime_ms"], sub["psnr"], label=method)

plt.xscale("log")
plt.xlabel("Runtime (ms, log scale)")
plt.ylabel("PSNR (dB)")
plt.title(f"PSNR vs Runtime — {image_path.stem}, σ={SIGMA}")
plt.legend()
plt.tight_layout()

fig_path = FIG_DIR / f"{DATASET}_{image_path.stem}_sigma{SIGMA}_psnr_runtime.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

fig_path


## 9. Visual comparison: best setting của mỗi method

In [ ]:
panels = [
    ("Original", clean, None),
    (f"Noisy σ={SIGMA}", noisy, psnr(clean, noisy)),
]

for method, item in best_images.items():
    panels.append((method, item["image"], item["psnr"]))

cols = 3
rows = int(np.ceil(len(panels) / cols))

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes = np.array(axes).reshape(rows, cols)

for ax in axes.flat:
    ax.axis("off")

for ax, (title, img, p) in zip(axes.flat, panels):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.axis("off")
    if p is None:
        ax.set_title(title)
    else:
        ax.set_title(f"{title}\nPSNR={p:.2f} dB")

plt.suptitle(f"Best Hyperparameter Visual Comparison — {image_path.stem}, σ={SIGMA}", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])

fig_path = FIG_DIR / f"{DATASET}_{image_path.stem}_sigma{SIGMA}_best_visual.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

fig_path


## 10. Gợi ý diễn giải kết quả

Bạn có thể dùng các ý sau trong report:

- **Gaussian**: kernel/sigma càng lớn thì ảnh càng mượt, nhưng dễ mất biên và texture.
- **NLM**: `h` càng lớn thì denoise càng mạnh nhưng dễ over-smooth; `search_size` lớn có thể cải thiện kết quả nhưng runtime tăng mạnh.
- **BM3D**: `sigma_psd` thấp hơn noise thật sẽ under-denoise; cao hơn quá nhiều sẽ over-smooth.
- **K-SVD**: dictionary lớn hơn và nhiều iteration hơn có thể cải thiện biểu diễn patch nhưng chi phí tính toán tăng rất rõ.

Câu chốt:
> Hyperparameter study cho thấy hiệu năng của denoising không chỉ phụ thuộc vào thuật toán, mà còn phụ thuộc rất mạnh vào cách chọn tham số.
